# AHS-KT × ASSIST2009 消融实验 Notebook

这本 Notebook 的目标是：

1. 使用 `/root/autodl-tmp/ahs-kt` 项目代码；
2. 使用 `ASSIST2009` 数据集；
3. 按照更接近论文写法的方式，对 `AHS-KT` 做 **3 组消融实验**；
4. 输出每组实验在 **多随机种子** 下的 `auc / acc / rmse / f1` 汇总结果。

## 为什么这里要做消融

从论文写作的第一性原理看，模型有效不应该只靠“整体结果更高”来证明。
更关键的是回答：
- 只有 target interaction 时效果怎样？
- 加入 difficulty 信息后，是否带来增益？
- 再加入 behavior cluster 后，是否继续提升？

所以这本 Notebook 采用 3 个逐步增强的版本：
- `target_only`：只保留 target interaction；
- `target_difficulty`：加入 difficulty；
- `target_difficulty_behavior_cluster`：再加入 behavior cluster。

## 这次为什么不用 `assist2009_v5`

因为消融实验最重要的是 **可比性**。

`v5` 已经不只是“多了某个模块”，还包含了更强的 difficulty 设计与数据版本调整。
如果直接拿 `v5` 做消融，容易把“模块贡献”和“训练配方变化”混在一起。

所以这里采用仓库已有的标准 `assist2009` ablation 配置：
- `ahskt_assist2009_ablation_target_only.json`
- `ahskt_assist2009_ablation_target_difficulty.json`
- `ahskt_assist2009_v2.json`

这样三组实验的数据、训练轮数、batch size 都一致，更适合论文里的 ablation section。


In [1]:
from pathlib import Path
import os
import sys
import json
import subprocess
from collections import defaultdict

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'

THREAD_ENV = {
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
    'GOTO_NUM_THREADS': '1',
}
for key, value in THREAD_ENV.items():
    os.environ[key] = value

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'scripts/run_assist2009_ablation.py').exists(), '缺少 assist2009 消融脚本'
assert (PROJECT_ROOT / 'data/assist2009_train_ahskt.npz').exists(), '缺少 assist2009 train bundle'
assert (PROJECT_ROOT / 'data/assist2009_valid_ahskt.npz').exists(), '缺少 assist2009 valid bundle'
assert (PROJECT_ROOT / 'data/assist2009_test_ahskt.npz').exists(), '缺少 assist2009 test bundle'

os.chdir(PROJECT_ROOT)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_ROOT =', SRC_ROOT)
print('线程环境变量 =')
for key in THREAD_ENV:
    print(f'  {key}={os.environ[key]}')


PROJECT_ROOT = /root/autodl-tmp/ahs-kt
SRC_ROOT = /root/autodl-tmp/ahs-kt/src
线程环境变量 =
  OMP_NUM_THREADS=1
  OPENBLAS_NUM_THREADS=1
  MKL_NUM_THREADS=1
  NUMEXPR_NUM_THREADS=1
  VECLIB_MAXIMUM_THREADS=1
  GOTO_NUM_THREADS=1


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error

from ahskt.config import load_config
from ahskt.data.dataset import load_bundle_from_config
from ahskt.models.ahs_kt import AHSKTModel

print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass


TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 参数区

这里默认采用 3 个随机种子：
- `2023`
- `2024`
- `2025`

这是因为仓库里已经有对应的 3-seed 消融结果，能直接复用并保持和现有实验对齐。

如果你想全量重跑：
- 把 `REUSE_EXISTING` 改成 `False`
- 或者把 `OUTPUT_PREFIX` 改成一个新的名字


In [3]:
SEEDS = [2023, 2024, 2025]
SEED_TEXT = ','.join(str(seed) for seed in SEEDS)
OUTPUT_PREFIX = 'assist2009_ablation_notebook_seed3'
REUSE_EXISTING = True

EXPERIMENTS = {
    'target_only': PROJECT_ROOT / 'configs/ahskt_assist2009_ablation_target_only.json',
    'target_difficulty': PROJECT_ROOT / 'configs/ahskt_assist2009_ablation_target_difficulty.json',
    'target_difficulty_behavior_cluster': PROJECT_ROOT / 'configs/ahskt_assist2009_v2.json',
}

EXPERIMENT_LABELS = {
    'target_only': '仅 target interaction',
    'target_difficulty': 'target + difficulty',
    'target_difficulty_behavior_cluster': 'target + difficulty + behavior cluster',
}

RAW_JSON_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_raw.json'
RAW_CSV_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_raw.csv'
AGG_JSON_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_aggregate.json'
AGG_CSV_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_aggregate.csv'
GENERATED_CONFIG_DIR = PROJECT_ROOT / 'outputs' / 'generated_configs' / OUTPUT_PREFIX

RAW_WITH_F1_JSON_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_raw_with_f1.json'
RAW_WITH_F1_CSV_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_raw_with_f1.csv'
AGG_WITH_F1_JSON_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_aggregate_with_f1.json'
AGG_WITH_F1_CSV_PATH = PROJECT_ROOT / 'outputs' / f'{OUTPUT_PREFIX}_aggregate_with_f1.csv'

print('SEEDS =', SEEDS)
print('OUTPUT_PREFIX =', OUTPUT_PREFIX)
print('REUSE_EXISTING =', REUSE_EXISTING)


SEEDS = [2023, 2024, 2025]
OUTPUT_PREFIX = assist2009_ablation_notebook_seed3
REUSE_EXISTING = True


In [4]:
config_rows = []
for experiment_name, config_path in EXPERIMENTS.items():
    payload = json.loads(config_path.read_text(encoding='utf-8'))
    config_rows.append({
        '实验': EXPERIMENT_LABELS[experiment_name],
        'config_path': str(config_path.relative_to(PROJECT_ROOT)),
        'use_target_interaction': payload['model'].get('use_target_interaction'),
        'use_difficulty_features': payload['model'].get('use_difficulty_features'),
        'use_behavior_features': payload['model'].get('use_behavior_features'),
        'use_behavior_cluster': payload['model'].get('use_behavior_cluster'),
        'epochs': payload['training']['epochs'],
        'batch_size': payload['training']['batch_size'],
        'learning_rate': payload['training']['learning_rate'],
        'dataset_train_path': payload['dataset']['train_path'],
    })

config_df = pd.DataFrame(config_rows)
print(config_df.to_string(index=False))


                                    实验                                              config_path  use_target_interaction  use_difficulty_features  use_behavior_features  use_behavior_cluster  epochs  batch_size  learning_rate              dataset_train_path
                  仅 target interaction       configs/ahskt_assist2009_ablation_target_only.json                    True                    False                  False                 False       3          16          0.001 data/assist2009_train_ahskt.npz
                   target + difficulty configs/ahskt_assist2009_ablation_target_difficulty.json                    True                     True                  False                 False       3          16          0.001 data/assist2009_train_ahskt.npz
target + difficulty + behavior cluster                         configs/ahskt_assist2009_v2.json                    True                     True                   True                  True       3          16          0.001 data

## 执行或复用消融实验

这里直接调用项目现成脚本：
- `scripts/run_assist2009_ablation.py`

它会针对 3 个实验配置 × 3 个随机种子，生成：
- raw 逐次结果
- aggregate 均值/方差汇总

默认使用 `--reuse-existing`，这样如果仓库里已经有结果，就不重复训练。


In [5]:
cmd = [
    sys.executable,
    'scripts/run_assist2009_ablation.py',
    '--seeds', SEED_TEXT,
    '--output-prefix', OUTPUT_PREFIX,
]
if REUSE_EXISTING:
    cmd.append('--reuse-existing')

print('执行命令:')
print(' '.join(cmd))
result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    env=os.environ.copy(),
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
assert RAW_JSON_PATH.exists(), f'缺少 raw 结果: {RAW_JSON_PATH}'
assert AGG_JSON_PATH.exists(), f'缺少 aggregate 结果: {AGG_JSON_PATH}'


执行命令:
/root/miniconda3/bin/python scripts/run_assist2009_ablation.py --seeds 2023,2024,2025 --output-prefix assist2009_ablation_notebook_seed3 --reuse-existing
[
  {
    "experiment": "target_difficulty",
    "seeds": "2023,2024,2025",
    "runs": 3,
    "best_valid_auc_mean": 0.7644408872465652,
    "best_valid_auc_std": 0.001022042828984121,
    "best_valid_auc_var": 1.0445715442778654e-06,
    "best_valid_auc_mean_pm_std": "0.764441±0.001022",
    "test_auc_mean": 0.7659861222827985,
    "test_auc_std": 0.0012486433547204282,
    "test_auc_var": 1.5591102272874849e-06,
    "test_auc_mean_pm_std": "0.765986±0.001249",
    "test_acc_mean": 0.7230630514587494,
    "test_acc_std": 0.0013710125352789089,
    "test_acc_var": 1.8796753718919014e-06,
    "test_acc_mean_pm_std": "0.723063±0.001371",
    "test_rmse_mean": 0.43321511149406433,
    "test_rmse_std": 0.0004933948508615572,
    "test_rmse_var": 2.434384788566983e-07,
    "test_rmse_mean_pm_std": "0.433215±0.000493",
    "test_loss

In [6]:
raw_rows = json.loads(RAW_JSON_PATH.read_text(encoding='utf-8'))
agg_rows = json.loads(AGG_JSON_PATH.read_text(encoding='utf-8'))

raw_df = pd.DataFrame(raw_rows)
agg_df = pd.DataFrame(agg_rows)

agg_display_df = pd.DataFrame({
    '实验': agg_df['experiment'].map(EXPERIMENT_LABELS),
    '随机种子': agg_df['seeds'],
    'runs': agg_df['runs'],
    'test_auc(mean±std)': agg_df['test_auc_mean_pm_std'],
    'test_acc(mean±std)': agg_df['test_acc_mean_pm_std'],
    'test_rmse(mean±std)': agg_df['test_rmse_mean_pm_std'],
})

print('AHS-KT assist2009 消融汇总（原生指标）')
print(agg_display_df.to_string(index=False))


AHS-KT assist2009 消融汇总（原生指标）
                                    实验           随机种子  runs test_auc(mean±std) test_acc(mean±std) test_rmse(mean±std)
                   target + difficulty 2023,2024,2025     3  0.765986±0.001249  0.723063±0.001371   0.433215±0.000493
target + difficulty + behavior cluster 2023,2024,2025     3  0.767911±0.002927  0.725865±0.002152   0.431416±0.001342
                  仅 target interaction 2023,2024,2025     3  0.779085±0.000865  0.731350±0.000441   0.424948±0.000384


## 补充 F1 指标

项目现成 ablation 脚本默认不汇总 `f1`。

为了让 Notebook 更完整，这里会：
- 读取每个 seed 的配置与 metrics；
- 从 checkpoint 恢复模型；
- 在测试集上重新做一次推理；
- 补出每个 run 的 `f1`；
- 再对 `f1` 做 mean / std 聚合。


In [7]:
def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)


def compute_run_metrics_with_f1(experiment_name, seed, metrics_path):
    config_path = GENERATED_CONFIG_DIR / f'{experiment_name}_s{seed}.json'
    assert config_path.exists(), f'缺少生成配置: {config_path}'
    metrics_payload = json.loads(Path(metrics_path).read_text(encoding='utf-8'))
    config = load_config(config_path, project_root=PROJECT_ROOT)
    _, _, test_bundle = load_bundle_from_config(config)

    model = AHSKTModel(config.model)
    sample_batch = next(iter(test_bundle.to_tf_dataset(batch_size=1, shuffle=False)))
    _ = model(sample_batch, training=False)

    checkpoint = tf.train.Checkpoint(model=model)
    checkpoint.restore(metrics_payload['checkpoint_path']).expect_partial()

    test_targets, test_predictions = collect_targets_and_predictions(
        model=model,
        bundle=test_bundle,
        batch_size=config.training.batch_size,
    )
    test_binary_predictions = (test_predictions > 0.5).astype(int)

    return {
        'experiment': experiment_name,
        'seed': int(seed),
        'task_name': metrics_payload['task_name'],
        'best_epoch': int(metrics_payload['best_epoch']),
        'best_valid_auc': float(metrics_payload['best_valid_auc']),
        'test_auc': float(roc_auc_score(test_targets, test_predictions)),
        'test_acc': float(accuracy_score(test_targets, test_binary_predictions)),
        'test_rmse': float(mean_squared_error(test_targets, test_predictions, squared=False)),
        'test_f1': float(f1_score(test_targets, test_binary_predictions)),
        'test_loss': float(metrics_payload['test_metrics']['loss']),
        'num_test_points': int(len(test_targets)),
        'metrics_path': str(metrics_path),
        'checkpoint_path': str(metrics_payload['checkpoint_path']),
    }


raw_with_f1_rows = []
for row in raw_rows:
    raw_with_f1_rows.append(
        compute_run_metrics_with_f1(
            experiment_name=row['experiment'],
            seed=int(row['seed']),
            metrics_path=row['metrics_path'],
        )
    )

raw_with_f1_df = pd.DataFrame(raw_with_f1_rows).sort_values(by=['experiment', 'seed']).reset_index(drop=True)
raw_with_f1_df.to_csv(RAW_WITH_F1_CSV_PATH, index=False)
RAW_WITH_F1_JSON_PATH.write_text(json.dumps(raw_with_f1_rows, ensure_ascii=False, indent=2), encoding='utf-8')

print('逐次结果（含 F1）已保存到:')
print(' ', RAW_WITH_F1_JSON_PATH)
print(' ', RAW_WITH_F1_CSV_PATH)
print(raw_with_f1_df[['experiment', 'seed', 'test_auc', 'test_acc', 'test_rmse', 'test_f1']].to_string(index=False))


2026-04-07 11:47:54.718713: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-07 11:47:55.435449: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:57:00.0, compute capability: 8.6
2026-04-07 11:47:56.286977: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-04-07 11:47:57.059138: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8200


逐次结果（含 F1）已保存到:
  /root/autodl-tmp/ahs-kt/outputs/assist2009_ablation_notebook_seed3_raw_with_f1.json
  /root/autodl-tmp/ahs-kt/outputs/assist2009_ablation_notebook_seed3_raw_with_f1.csv
                        experiment  seed  test_auc  test_acc  test_rmse  test_f1
                 target_difficulty  2023  0.765567  0.724931   0.432846 0.794654
                 target_difficulty  2024  0.767681  0.721678   0.433912 0.783660
                 target_difficulty  2025  0.764710  0.722580   0.432887 0.792295
target_difficulty_behavior_cluster  2023  0.764515  0.723546   0.433062 0.793227
target_difficulty_behavior_cluster  2024  0.771660  0.728731   0.429774 0.795910
target_difficulty_behavior_cluster  2025  0.767560  0.725317   0.431410 0.794140
                       target_only  2023  0.777917  0.731629   0.425491 0.803841
                       target_only  2024  0.779985  0.730727   0.424682 0.794181
                       target_only  2025  0.779354  0.731693   0.424671 0.799200


In [8]:
def metric_summary(values):
    array = np.asarray(list(values), dtype=np.float64)
    return {
        'mean': float(np.mean(array)),
        'std': float(np.std(array, ddof=0)),
        'var': float(np.var(array, ddof=0)),
        'mean_pm_std': f'{float(np.mean(array)):.6f}±{float(np.std(array, ddof=0)):.6f}',
    }


aggregate_with_f1_rows = []
for experiment_name, group_df in raw_with_f1_df.groupby('experiment', sort=True):
    auc_stats = metric_summary(group_df['test_auc'])
    acc_stats = metric_summary(group_df['test_acc'])
    rmse_stats = metric_summary(group_df['test_rmse'])
    f1_stats = metric_summary(group_df['test_f1'])
    valid_auc_stats = metric_summary(group_df['best_valid_auc'])
    aggregate_with_f1_rows.append({
        'experiment': experiment_name,
        'seeds': ','.join(str(seed) for seed in group_df['seed'].tolist()),
        'runs': int(len(group_df)),
        'best_valid_auc_mean': valid_auc_stats['mean'],
        'best_valid_auc_std': valid_auc_stats['std'],
        'best_valid_auc_mean_pm_std': valid_auc_stats['mean_pm_std'],
        'test_auc_mean': auc_stats['mean'],
        'test_auc_std': auc_stats['std'],
        'test_auc_mean_pm_std': auc_stats['mean_pm_std'],
        'test_acc_mean': acc_stats['mean'],
        'test_acc_std': acc_stats['std'],
        'test_acc_mean_pm_std': acc_stats['mean_pm_std'],
        'test_rmse_mean': rmse_stats['mean'],
        'test_rmse_std': rmse_stats['std'],
        'test_rmse_mean_pm_std': rmse_stats['mean_pm_std'],
        'test_f1_mean': f1_stats['mean'],
        'test_f1_std': f1_stats['std'],
        'test_f1_mean_pm_std': f1_stats['mean_pm_std'],
    })

aggregate_with_f1_df = pd.DataFrame(aggregate_with_f1_rows).sort_values(by='experiment').reset_index(drop=True)
aggregate_with_f1_df.to_csv(AGG_WITH_F1_CSV_PATH, index=False)
AGG_WITH_F1_JSON_PATH.write_text(json.dumps(aggregate_with_f1_rows, ensure_ascii=False, indent=2), encoding='utf-8')

final_display_df = pd.DataFrame({
    '实验': aggregate_with_f1_df['experiment'].map(EXPERIMENT_LABELS),
    '随机种子': aggregate_with_f1_df['seeds'],
    'runs': aggregate_with_f1_df['runs'],
    'AUC(mean±std)': aggregate_with_f1_df['test_auc_mean_pm_std'],
    'ACC(mean±std)': aggregate_with_f1_df['test_acc_mean_pm_std'],
    'RMSE(mean±std)': aggregate_with_f1_df['test_rmse_mean_pm_std'],
    'F1(mean±std)': aggregate_with_f1_df['test_f1_mean_pm_std'],
})

print('AHS-KT assist2009 消融汇总（补充 F1 后）')
print(final_display_df.to_string(index=False))
print()
print('聚合结果已保存到:')
print(' ', AGG_WITH_F1_JSON_PATH)
print(' ', AGG_WITH_F1_CSV_PATH)


AHS-KT assist2009 消融汇总（补充 F1 后）
                                    实验           随机种子  runs     AUC(mean±std)     ACC(mean±std)    RMSE(mean±std)      F1(mean±std)
                   target + difficulty 2023,2024,2025     3 0.765986±0.001249 0.723063±0.001371 0.433215±0.000493 0.790203±0.004726
target + difficulty + behavior cluster 2023,2024,2025     3 0.767911±0.002927 0.725865±0.002152 0.431416±0.001342 0.794426±0.001114
                  仅 target interaction 2023,2024,2025     3 0.779085±0.000865 0.731350±0.000441 0.424948±0.000384 0.799074±0.003945

聚合结果已保存到:
  /root/autodl-tmp/ahs-kt/outputs/assist2009_ablation_notebook_seed3_aggregate_with_f1.json
  /root/autodl-tmp/ahs-kt/outputs/assist2009_ablation_notebook_seed3_aggregate_with_f1.csv


In [9]:
base_row = aggregate_with_f1_df.set_index('experiment').loc['target_only']
delta_rows = []
for experiment_name in ['target_difficulty', 'target_difficulty_behavior_cluster']:
    row = aggregate_with_f1_df.set_index('experiment').loc[experiment_name]
    delta_rows.append({
        '实验': EXPERIMENT_LABELS[experiment_name],
        'ΔAUC(相对 target_only)': float(row['test_auc_mean'] - base_row['test_auc_mean']),
        'ΔACC(相对 target_only)': float(row['test_acc_mean'] - base_row['test_acc_mean']),
        'ΔRMSE(相对 target_only)': float(row['test_rmse_mean'] - base_row['test_rmse_mean']),
        'ΔF1(相对 target_only)': float(row['test_f1_mean'] - base_row['test_f1_mean']),
    })

delta_df = pd.DataFrame(delta_rows)
print('相对 target_only 的变化：')
print(delta_df.to_string(index=False))
print()

best_auc_row = aggregate_with_f1_df.sort_values(by='test_auc_mean', ascending=False).iloc[0]
best_rmse_row = aggregate_with_f1_df.sort_values(by='test_rmse_mean', ascending=True).iloc[0]
best_f1_row = aggregate_with_f1_df.sort_values(by='test_f1_mean', ascending=False).iloc[0]

print('最好 AUC 的实验 =', EXPERIMENT_LABELS[best_auc_row['experiment']], '->', round(float(best_auc_row['test_auc_mean']), 6))
print('最好 RMSE 的实验 =', EXPERIMENT_LABELS[best_rmse_row['experiment']], '->', round(float(best_rmse_row['test_rmse_mean']), 6))
print('最好 F1 的实验 =', EXPERIMENT_LABELS[best_f1_row['experiment']], '->', round(float(best_f1_row['test_f1_mean']), 6))


相对 target_only 的变化：
                                    实验  ΔAUC(相对 target_only)  ΔACC(相对 target_only)  ΔRMSE(相对 target_only)  ΔF1(相对 target_only)
                   target + difficulty             -0.013099             -0.008287               0.008267            -0.008871
target + difficulty + behavior cluster             -0.011174             -0.005485               0.006467            -0.004648

最好 AUC 的实验 = 仅 target interaction -> 0.779085
最好 RMSE 的实验 = 仅 target interaction -> 0.424948
最好 F1 的实验 = 仅 target interaction -> 0.799074


## 如何用于论文写作

这本 Notebook 适合直接支撑论文里的 ablation section，原因是：
- 同一数据集：`ASSIST2009`
- 同一训练轮数与 batch size：保证可比性
- 多随机种子：报告 `mean ± std`，而不是只报单次最好值
- 逐步加模块：能回答 difficulty 与 behavior cluster 是否真正带来收益

如果你下一步要把它写进论文，最自然的延伸是：
1. 再补一张 `ASSIST2012` 的同结构消融表；
2. 把 `assist2009` 与 `assist2012` 的结论并排写；
3. 解释为什么某些模块在不同数据集上收益不同。
